# 📓 Notebook 19 — Scheduling and Automation Orchestration

> **Module:** Software Engineering · **Estimated time:** 45–60 min · **Difficulty:** Intermediate

You have a package (NB18), the package can answer a question, run a model, parse a document. The last mile of "automation" is making the package run **without a human present** — every Monday at 6 a.m., every hour, every time a new file lands. This notebook closes that loop.

We will:

1. Build a small **task function** — the unit of work a scheduler runs.
2. Run it on a fake schedule **inside this notebook** so you see the loop work.
3. Add **logging, retries, idempotency, and notifications** — the four production essentials.
4. Show the exact **cron**, **systemd timer**, **GitHub Actions**, and **Prefect** snippets so you can pick a host and ship.

This is the notebook that turns *"a Python project"* into *"a working automation"*.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Write a **task function** that's safe to call repeatedly.
2. Use the **`schedule`** library for in-process scheduling (no extra infrastructure).
3. Add **structured logging** so a failed 3 a.m. run leaves a clear trace.
4. Make tasks **idempotent** so a re-run doesn't double-charge or double-send.
5. Wire up **email / Slack / webhook** notifications for failures.
6. Pick between **cron**, **systemd**, **GitHub Actions**, and **Prefect/Airflow**.
7. Set up **retries with backoff** at the task level.

## ✅ Prerequisites

Notebook 6 (functions), NB12 (APIs / HTTP), NB18 (project structure). NB10 (capstone) gives the scenario we'll automate.

## 1. What does "automation" actually mean?

```
   ┌───────────────────────────────────────────────────────────────────────┐
   │  TASK FUNCTION          ── one unit of work, runs end-to-end          │
   │     │                                                                 │
   │     ├──► fetches the data                                             │
   │     ├──► does the analysis                                            │
   │     ├──► writes the output                                            │
   │     └──► sends the notification                                       │
   │                                                                       │
   │  SCHEDULER             ── runs the task on a clock                    │
   │     │                                                                 │
   │     └──► cron / systemd / GitHub Actions / Prefect / Airflow / …      │
   │                                                                       │
   │  OBSERVABILITY         ── tells you when something went wrong         │
   │     │                                                                 │
   │     ├──► logs                                                          │
   │     ├──► alerts (Slack / email / PagerDuty)                            │
   │     └──► a dashboard                                                   │
   └───────────────────────────────────────────────────────────────────────┘
```

The task function is something *you* write (NB18-style). The scheduler is something you *configure*. The observability is what stops a silent failure from becoming a Monday-morning fire drill.

## 2. The task function — a runnable unit of work

In [ ]:
import json
import logging
import random
import time
from datetime import datetime
from pathlib import Path

# Configure logging once at the top of the script.
# In a real deployment the handler would be `logging.FileHandler('app.log')`.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("weekly_report")


def weekly_report_task(run_id: str | None = None) -> dict:
    """
    A self-contained scheduled task.

    1. Fetches yesterday's metrics (here mocked).
    2. Computes a tiny summary.
    3. Writes the result to disk.
    4. Returns a status dict the scheduler can log.
    """
    run_id = run_id or datetime.utcnow().strftime("%Y%m%d-%H%M%S")
    log.info(f"weekly_report START run_id={run_id}")

    try:
        # 1. Fetch (mocked here; in reality this would be an API call or DB query)
        random.seed(int(run_id[-4:]) if run_id[-4:].isdigit() else 0)
        tickets = random.randint(900, 1500)
        auto_rate = round(random.uniform(0.55, 0.85), 3)

        # 2. Compute
        manual_tickets = int(tickets * (1 - auto_rate))
        summary = {
            "run_id":      run_id,
            "tickets":     tickets,
            "auto_rate":   auto_rate,
            "manual":      manual_tickets,
            "generated":   datetime.utcnow().isoformat(timespec="seconds"),
        }

        # 3. Write
        out_path = Path("/tmp/reports") / f"{run_id}.json"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(json.dumps(summary, indent=2))
        log.info(f"wrote {out_path} ({tickets} tickets, auto={auto_rate:.0%})")

        return {"status": "ok", "summary": summary, "output": str(out_path)}

    except Exception as e:
        log.exception(f"weekly_report FAILED run_id={run_id}")
        return {"status": "error", "error": str(e), "run_id": run_id}


# Run it once, manually
result = weekly_report_task()
print(f"\nResult: {result['status']}")
print(json.dumps(result.get('summary', {}), indent=2))


**Three habits inside that function that are non-obvious:**

1. **`run_id` is an argument** with a sensible default — so a scheduler can pass a deterministic id (e.g. `2024-05-13`) and you can re-run a specific date.
2. **The outer `try / except` doesn't crash** — it returns a status dict. A scheduler can then `if result["status"] == "error": send_alert(...)`.
3. **The function logs at INFO** for normal events and `log.exception(...)` for failures (which captures the full traceback).

## 3. In-process scheduling with `schedule`

The `schedule` library is the simplest possible scheduler: it lives in your Python process and runs a function on a clock. Not for production, but perfect for **understanding the loop**.

```bash
pip install schedule
```

We'll simulate "every minute" by running a fast tick-loop for ~5 seconds.

In [ ]:
# In real code: `import schedule`. We re-implement it in 20 lines for offline runnability.
class TinyScheduler:
    """A minimal stand-in for the `schedule` library. Same calling pattern."""
    def __init__(self):
        self.jobs = []   # list of (next_run_ts, interval_s, fn)

    def every(self, interval_s):
        scheduler_self = self
        class _Job:
            def __init__(self): self.interval = interval_s
            def do(self, fn):
                scheduler_self.jobs.append([time.time() + self.interval, self.interval, fn])
                return self
        return _Job()

    def run_pending(self):
        now = time.time()
        for job in self.jobs:
            if now >= job[0]:
                job[2]()                          # call fn()
                job[0] = now + job[1]             # schedule next run


scheduler = TinyScheduler()

# Schedule the task to run every 1 second (for the demo)
scheduler.every(1.0).do(lambda: weekly_report_task())
print("Scheduler started. Running for 3 seconds...")

end_time = time.time() + 3.5
while time.time() < end_time:
    scheduler.run_pending()
    time.sleep(0.2)

print("\n✅ Scheduler stopped.")
print(f"Reports produced: {len(list(Path('/tmp/reports').glob('*.json')))}")


**The real `schedule` library** uses these same patterns:

```python
import schedule
schedule.every().monday.at("06:00").do(weekly_report_task)
schedule.every().hour.at(":15").do(hourly_sync)
schedule.every(15).minutes.do(health_check)

while True:
    schedule.run_pending()
    time.sleep(1)
```

> ⚠️ **Don't use `schedule` for anything important.** If the Python process crashes, your jobs stop. Use it for ad-hoc local automation; use cron/systemd/Prefect for anything that *matters*.

## 4. Idempotency — safe to re-run

A scheduled task **will** run twice eventually. A network blip, a manual re-run, a retry-after-failure. Your task must be **idempotent**: running it N times has the same effect as running it once.

In [ ]:
def idempotent_weekly_report(run_id: str) -> dict:
    """Same as before, but checks whether the output already exists before doing the work."""
    out_path = Path("/tmp/reports") / f"{run_id}.json"
    if out_path.exists():
        log.info(f"weekly_report SKIP run_id={run_id} (output already exists)")
        return {"status": "skipped", "run_id": run_id, "reason": "already-done"}
    return weekly_report_task(run_id=run_id)


# Run twice with the same run_id — the second time should be a no-op
print("First call  :", idempotent_weekly_report("2024-05-13")["status"])
print("Second call :", idempotent_weekly_report("2024-05-13")["status"])


**Other ways to make a task idempotent:**

- **Database upserts** instead of inserts. `INSERT ... ON CONFLICT DO UPDATE` is the SQL phrase to learn.
- **Use a deterministic `run_id`** (date, hour) as a primary key, not a random UUID.
- **Track "already-sent" notifications** in a small log table so a duplicate run doesn't re-email people.

> 🎯 **The pragmatic test.** *"If a tired engineer hits 'rerun' on this task at 11 p.m., does anything bad happen?"* If yes — make it idempotent before going home.

## 5. Retries with backoff — survive the flaky internet

In [ ]:
def with_retry(fn, max_attempts: int = 4, base_delay: float = 0.3):
    """Run `fn()` with exponential backoff. Returns the result or raises the final exception."""
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except Exception as e:
            if attempt == max_attempts:
                log.error(f"giving up after {attempt} attempts: {type(e).__name__}: {e}")
                raise
            wait = base_delay * 2 ** (attempt - 1)
            log.warning(f"attempt {attempt} failed ({type(e).__name__}); retrying in {wait:.2f}s")
            time.sleep(wait)


# A flaky function: fails for the first two calls, then succeeds.
class _Flaky:
    def __init__(self): self.calls = 0
    def __call__(self):
        self.calls += 1
        if self.calls < 3:
            raise ConnectionError("network blip")
        return "OK"


result = with_retry(_Flaky(), max_attempts=5, base_delay=0.1)
print(f"\nFinal result: {result}")


> 💡 **Where to put retries.** *Inside the task function*, around the calls that can fail (HTTP, DB, file I/O). Wrap an entire task in retry only if you've made it idempotent — otherwise a duplicate run will run the side-effects twice.

## 6. Notifications — fail loud

A task that fails silently at 3 a.m. is much worse than one that fails loudly. Send a notification on errors *and* on first-time success (so you know it's wired up).

Three common channels:

| Channel | When to use | Library / endpoint |
|---|---|---|
| **Email** | Daily summaries, low-priority alerts | `smtplib`, SendGrid API, AWS SES |
| **Slack** | Real-time team alerts | Incoming webhooks (free) |
| **PagerDuty / Opsgenie** | Wake someone up | Paid; integrates with the above |

Demo — a no-op Slack webhook simulator:

In [ ]:
def send_slack_alert(text: str, webhook_url: str | None = None) -> bool:
    """Post to a Slack incoming webhook (or print if no URL is set)."""
    payload = {"text": text}
    if not webhook_url:
        log.info(f"[SLACK MOCK] {payload}")
        return True
    # Real implementation:
    # import requests
    # r = requests.post(webhook_url, json=payload, timeout=8)
    # r.raise_for_status()
    return True


def task_with_alerts(run_id: str | None = None):
    result = idempotent_weekly_report(run_id or datetime.utcnow().strftime("%Y-%m-%d"))
    if result["status"] == "error":
        send_slack_alert(f"🚨 weekly_report FAILED for run_id={result['run_id']}: {result.get('error')}")
    elif result["status"] == "ok":
        s = result["summary"]
        send_slack_alert(f"✅ Weekly report: {s['tickets']} tickets, "
                         f"auto = {s['auto_rate']:.0%} (run_id {s['run_id']})")
    return result


task_with_alerts(run_id="2024-05-20")


> ⚠️ **Don't drown your team in green-pings.** Send "success" notifications during the first week after a new schedule lands, then turn them off. Long-term, *only* alert on failure — your colleagues will thank you.

## 7. Picking a host — cron, systemd, GitHub Actions, Prefect

You don't need a fancy orchestrator until you really do. Here's a decision tree:

```
                          ┌── on your laptop, low-frequency ──→ cron
                          │
   Where does the task    ┼── on a Linux server, low-frequency ──→ systemd timer
   actually run?         │
                          ├── on GitHub already, weekly / on-PR ──→ GitHub Actions
                          │
                          └── multiple tasks, dependencies between them ──→ Prefect / Airflow / Dagster
```

### `cron` — the classic

```bash
# /etc/cron.d/weekly-report (or `crontab -e`)
# m  h  dom mon dow   command
0 6 * * 1   /usr/bin/env -i HOME=/home/data ./.venv/bin/python -m costkit.weekly_report
```

The cryptic `* * * * *` left-to-right: minute (0–59), hour (0–23), day-of-month (1–31), month (1–12), day-of-week (0–6, Sunday=0). `0 6 * * 1` means *Monday at 06:00*.

### `systemd` — better than cron on modern Linux

Two files: a `.service` unit (what to run) and a `.timer` unit (when). systemd handles logs, retries, environment, and dependencies far better than cron — and `journalctl -u weekly-report` shows your task's logs in one place.

```ini
# /etc/systemd/system/weekly-report.service
[Service]
ExecStart=/opt/costkit/.venv/bin/python -m costkit.weekly_report

# /etc/systemd/system/weekly-report.timer
[Timer]
OnCalendar=Mon 06:00
Persistent=true        # run even if the box was off at 6 a.m.
```

### GitHub Actions — when the code already lives on GitHub

```yaml
# .github/workflows/weekly-report.yml
on:
  schedule:
    - cron: "0 6 * * 1"        # Mondays at 06:00 UTC
  workflow_dispatch:           # also allow manual triggers
jobs:
  weekly:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -e ".[dev]"
      - run: python -m costkit.weekly_report
        env:
          SLACK_WEBHOOK_URL: ${{ secrets.SLACK_WEBHOOK_URL }}
```

Free for public repos and most private use. **Caveat**: scheduled workflows are *best-effort* — GitHub may delay them by up to ~15 min under load.

### `Prefect` (or Airflow, Dagster) — multi-task workflows

The moment you have **more than one task** with **dependencies** between them (task A produces a file that task B consumes), a true orchestrator earns its keep.

```python
# prefect_flow.py
from prefect import flow, task

@task
def fetch_tickets():    ...

@task
def build_report(data): ...

@task
def send_alert(report): ...

@flow(name="weekly-report")
def weekly():
    data   = fetch_tickets()
    report = build_report(data)
    send_alert(report)

weekly.serve(name="weekly-report", cron="0 6 * * 1")
```

Prefect gives you a UI, retries with backoff, parallelism, parameterised runs, and a state machine. Airflow does the same with a different API. Use these when *workflow complexity* (not just *schedule complexity*) is the problem.

## 8. Logging — what to record

Three log lines per task at minimum:

1. **START** at the top: `task=X run_id=Y started`.
2. **Step events** at INFO: `fetched 1234 tickets`, `wrote report.json`.
3. **Final status** at the bottom: `task=X run_id=Y status=ok duration=12.3s` (or `status=error`).

Put logs to a **rotating file** (`logging.handlers.RotatingFileHandler`) so they don't fill the disk. Push them to a central system (Datadog, CloudWatch, Loki) when scale demands it.

> 💡 **Structured logs win.** A line like `auto_rate=0.78 tickets=1284 run_id=2024-05-13` is searchable and machine-parseable. The `structlog` package gives you JSON logs with one import — worth it for anything beyond a hobby project.

## 9. Putting it together — the production-shape skeleton

In [ ]:
def run_production_task(task_fn, run_id: str, alert_fn=send_slack_alert,
                         max_retries: int = 3) -> dict:
    """
    The full production-shape wrapper around one task:

    - timestamps the run
    - calls the task with retry+backoff
    - turns exceptions into structured error results
    - alerts on failure
    - returns the duration so a dashboard can plot it
    """
    started = time.perf_counter()
    log.info(f"--- run_production_task task={task_fn.__name__} run_id={run_id} START ---")
    try:
        result = with_retry(lambda: task_fn(run_id=run_id), max_attempts=max_retries)
        duration = time.perf_counter() - started
        result["duration_s"] = round(duration, 3)
        log.info(f"--- DONE run_id={run_id} status={result['status']} dur={duration:.2f}s ---")
        if result.get("status") == "error" and alert_fn:
            alert_fn(f"🚨 {task_fn.__name__} FAILED for run_id={run_id}")
        return result
    except Exception as e:
        duration = time.perf_counter() - started
        log.exception(f"--- FAIL run_id={run_id} dur={duration:.2f}s ---")
        if alert_fn:
            alert_fn(f"🚨 {task_fn.__name__} crashed for run_id={run_id}: {type(e).__name__}: {e}")
        return {"status": "error", "error": str(e), "run_id": run_id,
                "duration_s": round(duration, 3)}


# Run it twice with the same run_id — the second is a fast skip
out1 = run_production_task(idempotent_weekly_report, "2024-05-27")
out2 = run_production_task(idempotent_weekly_report, "2024-05-27")
print(f"\nRun 1: {out1['status']}, {out1['duration_s']}s")
print(f"Run 2: {out2['status']}, {out2['duration_s']}s   (idempotent → fast)")


**That wrapper is the entire blueprint.** Swap `task_fn` for any function you wrote in the previous notebooks (the capstone report, the embeddings re-index, the document-extraction batch) and you have an automation.

## 10. Common pitfalls

| Pitfall | Symptom | Fix |
|---|---|---|
| Cron jobs that work in the terminal but fail under cron | path/env differences | Always use absolute paths; set `PATH` and any env vars at the top of the cron line |
| The schedule runs but the data isn't fresh | timezone mismatch | Pin tasks to UTC; convert in the output only |
| Two runs collide | overlapping cron + slow task | Use a lockfile (`fcntl.flock`, `filelock`) or `flock(1)` |
| The task quietly stopped a month ago | nobody's watching | A daily "I'm alive" heartbeat to a monitoring service |
| Secret rotation breaks everything overnight | hard-coded credentials | Load secrets at runtime; rotate via the secret manager, not the code |
| Logs fill the disk | no rotation | `RotatingFileHandler` with sensible `maxBytes` and `backupCount` |

## 🧪 Practice exercises

### Exercise 1 — Add a duration alert

Modify `run_production_task` so it sends a warning alert (different from an error alert) when the task takes more than `slow_threshold_s` seconds. Test it by setting the threshold to 0.001 — every successful run should now also trigger a "slow" alert.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def run_production_task_v2(task_fn, run_id, alert_fn=send_slack_alert,
                            max_retries=3, slow_threshold_s=10.0):
    started = time.perf_counter()
    try:
        result = with_retry(lambda: task_fn(run_id=run_id), max_attempts=max_retries)
    except Exception as e:
        if alert_fn:
            alert_fn(f"🚨 {task_fn.__name__} crashed: {e}")
        raise
    duration = time.perf_counter() - started
    result["duration_s"] = duration
    if duration > slow_threshold_s:
        alert_fn(f"⚠ {task_fn.__name__} slow: {duration:.2f}s > {slow_threshold_s}s")
    return result


run_production_task_v2(idempotent_weekly_report, "2024-06-03",
                       slow_threshold_s=0.001)
```

**Why this matters.** *Slow* runs are the early warning sign of a future *failed* run. Wiring up a duration alert is much cheaper than firefighting later.
</details>

### Exercise 2 — A lock-file guard

Write a `with_lock(path)` context manager that creates a lockfile at `path`, raises `RuntimeError("already locked")` if it already exists, and removes it on exit. Wrap your task in it so two concurrent runs can't collide.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def with_lock(path: str):
    lockfile = Path(path)
    if lockfile.exists():
        raise RuntimeError(f"already locked: {path}")
    lockfile.touch()
    try:
        yield
    finally:
        lockfile.unlink(missing_ok=True)


# Demo
LOCK_PATH = "/tmp/weekly_report.lock"
with with_lock(LOCK_PATH):
    weekly_report_task(run_id="2024-06-10")

# A second concurrent call would now raise — uncomment to see:
# with with_lock(LOCK_PATH): ...
```

**In production** you want `filelock.FileLock` (`pip install filelock`) which is process-safe and cross-platform. The 8-line version above is correct enough for a single-machine scheduler.
</details>

### Exercise 3 — Schedule three tasks together

Use `TinyScheduler` (or the real `schedule` library) to register three tasks at three different intervals (e.g. 0.5s, 1s, 1.5s for the demo) and run them for 3 seconds. Count how many times each ran.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
counts = {"fast": 0, "medium": 0, "slow": 0}

def make_counter(name):
    def fn():
        counts[name] += 1
    return fn

sched = TinyScheduler()
sched.every(0.5).do(make_counter("fast"))
sched.every(1.0).do(make_counter("medium"))
sched.every(1.5).do(make_counter("slow"))

end = time.time() + 3.2
while time.time() < end:
    sched.run_pending()
    time.sleep(0.05)

print(counts)
```

`fast` should run ~6 times, `medium` ~3, `slow` ~2. Production schedulers manage thousands of tasks the same way — same loop, different infrastructure underneath.
</details>

### Exercise 4 — Debug me 🐞

The task below is meant to update a daily counter file. Run it twice in a row — the count goes from `1` to `2`. But run it once a day under cron and it occasionally **resets to 1** mid-week. What's the likely cause?

```python
def increment_counter():
    path = Path("/tmp/counter.txt")
    if path.exists():
        n = int(path.read_text()) + 1
    else:
        n = 1
    path.write_text(str(n))
```

<details>
<summary>💡 <b>Solution</b></summary>

A few likely causes — all real production bugs:

1. **`/tmp` is cleared on reboot** on most Linux systems. If the host restarts overnight, your counter file is gone. Fix: put state in a stable directory like `~/.local/share/myapp/` (or a database).
2. **Race condition** if two runs ever overlap: both read `1`, both write `2`. Use a lockfile (Exercise 2) or atomic write (`path.write_text(...)` is roughly atomic on POSIX, but rename-into-place is safer).
3. **Permissions changed** — a different user ran the task and `/tmp/counter.txt` was owned by someone else. The script fell back to "create new".

**Lesson.** Treat *anything in `/tmp`* as ephemeral. For real state, use a real store (a database, a known persistent directory, S3). The bug looks like the code; it's actually the assumption about where state lives.
</details>

## 🎁 Bonus mini-project — A daily metrics digest

Build a function `daily_digest()` that:

1. Reads all JSON files from `/tmp/reports/`.
2. Aggregates `tickets` and `auto_rate` across the runs.
3. Sends a (mock) Slack summary: `"Daily digest: N runs, total X tickets, mean auto-rate Y%"`.
4. Returns a `dict` with the aggregated numbers.

Then schedule it to run once and verify.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def daily_digest():
    rows = []
    for f in sorted(Path("/tmp/reports").glob("*.json")):
        try:
            rows.append(json.loads(f.read_text()))
        except json.JSONDecodeError:
            log.warning(f"skipping malformed: {f}")

    if not rows:
        send_slack_alert("ℹ️ Daily digest: no runs found.")
        return {"n_runs": 0}

    total_tickets = sum(r["tickets"] for r in rows)
    mean_auto    = sum(r["auto_rate"] for r in rows) / len(rows)
    summary = f"📅 Daily digest: {len(rows)} runs, " \
              f"{total_tickets:,} tickets, mean auto-rate {mean_auto:.1%}"
    send_slack_alert(summary)
    return {"n_runs": len(rows), "total_tickets": total_tickets,
            "mean_auto": round(mean_auto, 3)}

print(daily_digest())
```

**What you built.** A *report on the reports* — the meta-layer every production data team eventually adds. Schedule this nightly (after all the per-team tasks finish) and you get a one-line health snapshot of your entire automation portfolio every morning.
</details>

## 🧠 Key takeaways

1. **A task is a function that runs end-to-end** — fetch, compute, write, notify.
2. **Idempotent tasks are safe to re-run.** Use deterministic run-ids; check before doing work.
3. **Retries + backoff** at the task boundary; lockfiles to prevent concurrent collisions.
4. **Log START + steps + END/FAIL.** Use rotating file handlers in production.
5. **Alert on failure, sparingly on success.** Slack webhooks are free; email is universal.
6. **Pick the host that fits the job:** cron → systemd → GitHub Actions → Prefect/Airflow.
7. **State in `/tmp` is ephemeral.** Real state lives in a database or a known durable path.
8. The production wrapper around any task is ~30 lines — and it's the same for every project.

## ✅ Self-assessment

- [ ] Write a task function that returns a structured status dict
- [ ] Make a task idempotent with a deterministic run-id and an existence check
- [ ] Wrap a flaky operation in retry-with-backoff
- [ ] Use a lockfile to prevent concurrent runs
- [ ] Wire up structured logging at INFO + WARNING + ERROR
- [ ] Write a cron line or a systemd timer for a known schedule
- [ ] Decide when to upgrade from cron to a real orchestrator

## 🚀 You've reached the end

Congratulations — you've completed the 19-notebook journey. You started with `print("Hello")` and you've finished with a packaged, tested, scheduled, observable AI automation. From here, **what you build is up to you** — and the patterns you've internalised will carry across every Python tool and every AI service that lands tomorrow.

> 🚀 Open a notebook. Edit one number. Re-run. Iterate. *That* is the whole craft.